# ULTRON owner setup — preflight only

**This notebook does not train.** It does not instantiate the model, create an optimizer, generate or download a dataset, perform optimizer steps, or dispatch remote jobs.

Purpose: confirm the repository is present, load the website-exported schema-v1 runtime configuration, and independently calculate the foundation model's parameter counts from configuration formulas. Full dataset preflight, sampler/DataLoader construction, and clean-runtime resume are intentionally not included until the dataset manifest/split handoff is implemented and reviewed.

The config file is untrusted input. Its authorization flags must remain false.

In [ ]:
from pathlib import Path
import json
import platform
import sys

repo_root = Path.cwd()
if not (repo_root / "ultron_core").is_dir():
    raise FileNotFoundError("Run this notebook from the ULTRON repository root (ultron_core/ must exist).")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
print({"python": platform.python_version(), "platform": platform.platform(), "repo_root": str(repo_root)})

## Load the website-exported config

On the device/browser, export `ultron-runtime-config.json` from ULTRON Settings, then upload it to this notebook runtime using its file-upload UI. Enter the runtime-local path below. This is a manual file handoff; the website does not connect to this notebook.

In [ ]:
from ultron_core.runtime_config import RuntimeConfig, MAX_CONFIG_JSON_BYTES

config_path = Path(input("Runtime-local path to ultron-runtime-config.json: ").strip()).expanduser()
if not config_path.is_file():
    raise FileNotFoundError(f"Config file not found: {config_path}")
config_bytes = config_path.read_bytes()
if len(config_bytes) > MAX_CONFIG_JSON_BYTES:
    raise ValueError("Config exceeds the schema's 16 KiB limit.")
runtime_config = RuntimeConfig.from_json(config_bytes.decode("utf-8"))
assert runtime_config.training_authorized is False
assert runtime_config.remote_execution_enabled is False
print(json.dumps(runtime_config.to_dict(), indent=2, sort_keys=True))

## Capacity formula check — no model allocation

This constructs only the small immutable configuration object. The count functions use parameter-shape formulas; they do not instantiate `NeuroSparseModel` or allocate its weights.

In [ ]:
from ultron_core.config import NeuroSparseConfig, count_parameters, count_total_parameters, MIN_FOUNDATION_TRAINABLE_PARAMETERS

foundation_config = NeuroSparseConfig.foundation()
trainable_count = count_parameters(foundation_config)
total_count = count_total_parameters(foundation_config)
assert trainable_count >= MIN_FOUNDATION_TRAINABLE_PARAMETERS
assert total_count >= trainable_count
assert foundation_config.vocab_size == 240
print({
    "total_parameters_by_formula": total_count,
    "trainable_parameters_by_formula": trainable_count,
    "minimum_trainable_parameters": MIN_FOUNDATION_TRAINABLE_PARAMETERS,
    "sequence_length_limit": foundation_config.max_sequence_length,
    "runtime_budget_gib_requested": runtime_config.memory_budget_gib,
    "batch_size_requested": runtime_config.batch_size,
    "sequence_length_requested": runtime_config.sequence_length,
    "gradient_accumulation_requested": runtime_config.gradient_accumulation_steps,
})

## Stop point

Expected result: `CONFIG_VALIDATED / CAPACITY_FORMULA_CHECKED / PREFLIGHT_ONLY`.

This is **not** a full dataset preflight. No dataset has been loaded or generated, no train/validation/test split has been inspected, no model weights have been allocated, no checkpoint has been restored, no sampler or DataLoader has been created, and no training has occurred. Do not mark the training gate complete based on this notebook.

Next engineering stage: implement a read-only, manifest-bound loader for an existing dataset and split contract; then wire the resumable sampler and checkpoint callback metadata. Clean-runtime equivalence requires separate authorization for a tiny optimizer-step test.